In [1]:
import os

from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [2]:
load_dotenv()

True

In [3]:
current_dir = os.getcwd()
persistent_directory = os.path.join(current_dir, "db", "chroma_db_with_metadata")

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [5]:
db = Chroma(persist_directory=persistent_directory,
            embedding_function=embeddings)

In [9]:
query = "How can I learn more about LangChain?"

# Retrieve relevant documents based on the query
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
relevant_docs = retriever.invoke(query)

In [10]:
# Display the relevant results with metadata
print("\n--- Relevant Documents ---")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")

# Combine the query and the relevant document contents
combined_input = (
    "Here are some documents that might help answer the question: "
    + query
    + "\n\nRelevant Documents:\n"
    + "\n\n".join([doc.page_content for doc in relevant_docs])
    + "\n\nPlease provide an answer based only on the provided documents. If the answer is not found in the documents, respond with 'I'm not sure'."
)


--- Relevant Documents ---


In [11]:
model = ChatOpenAI(model="gpt-4o")
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]
result = model.invoke(messages)
print(result.content)

I'm not sure.
